# LAB | API Calling with JSON Extra

## Scenario

You're continuing your work on the automated product listing generator. Clients now want to provide product information via JSON API requests instead of only using images. Your task is to build a robust system that validates incoming JSON data using Pydantic, ensures data quality, and processes client requests reliably.

This lab builds on your previous ChatGPT API work and adds data validation capabilities.
**Refactored for Python for Low Code:** this version applies helper functions, modular design, explicit error handling, retry logic, and logging.


## Learning Objectives

- Understand JSON schema validation
- Use Pydantic for data validation
- Create Pydantic models for structured data
- Validate API request payloads
- Handle validation errors gracefully
- Integrate validation with API processing
- Build a complete API workflow with validation

**Estimated Time:** 120-150 minutes

## Prerequisites

- Completion of Lab M1.05 (API Calling to ChatGPT)
- Understanding of JSON data format
- Basic knowledge of Python classes
- Familiarity with error handling

## Environment Setup

Run the next cell to install the required libraries into the same Python environment used by this notebook kernel.


In [ ]:
%pip install -r requirements.txt


## Generated Artifacts

This notebook writes sample and output files into `data/` and writes logs into `logs/` when applicable. Those folders contain generated artifacts, not hand-written source files.


## What You'll Build

- Pydantic models for product data validation
- A JSON validation system
- Integration with your existing ChatGPT API workflow
- Error handling for invalid data
- A complete validated API pipeline

### Why this matters

In real-world applications, you cannot assume incoming data is correct. Validation prevents errors, improves security, and ensures data quality. Pydantic is a widely used Python library for validation and is commonly used in frameworks like FastAPI.

## Success Criteria

- Create Pydantic models for product data
- Validate JSON input successfully
- Handle validation errors appropriately
- Integrate validation with API processing
- Process validated requests end-to-end

## Background Story

Your product listing generator is working well, but now clients want to send product information via API requests. However, some clients send incomplete data, wrong data types, missing fields, or invalid values such as negative prices and empty names.

You need to validate all incoming data before processing to ensure:

- Data quality and consistency
- System reliability
- Better error messages for clients
- Prevention of API errors downstream

Pydantic will help you create validation rules and automatically check all incoming data.

# Step 1: Setting Up Pydantic

**Objective:** Install Pydantic and understand its basic usage.

### What to do

- Install Pydantic
- Create a simple validation example
- Understand how Pydantic works

### Required package

```bash
pip install pydantic
```

### Suggested starter ideas

- Create a simple `BaseModel`
- Add fields like `name`, `price`, and `quantity`
- Add validators for positive numeric values

### Expected outcome

You should see how Pydantic validates data and raises errors for invalid input.

### Checkpoint

Verify that Pydantic is installed and basic validation works.

In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator
from openai import OpenAI, APIError, APIConnectionError, APITimeoutError
from typing import Optional, List
from datetime import datetime
from pathlib import Path
import traceback
import logging
import time
import json
import os

print("=" * 60)
print("STEP 1 | SETUP, IMPORTS, AND LOGGING")
print("=" * 60)

DEFAULT_MODEL = "gpt-4o-2024-08-06"
DATA_DIR = Path("data")
LOG_DIR = Path("logs")
DATA_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / f"product_generator_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("product_generator")


def build_error_message(function_name: str, error_type: str, location: str, message: str, suggestion: str) -> str:
    """Create a consistent, explicit error message."""

    return (
        f"ERROR in {function_name}(): {error_type}\n"
        f"  Location: {location}\n"
        f"  Message: {message}\n"
        f"  Suggestion: {suggestion}"
    )


class SimpleProduct(BaseModel):
    """Small validation example to verify the notebook environment."""

    name: str = Field(min_length=2)
    price: float = Field(gt=0)
    quantity: int = Field(default=1, ge=1)

    @field_validator("name")
    @classmethod
    def clean_name(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Name cannot be empty")
        return value


print("\nValidation smoke test:")
try:
    example = SimpleProduct(name="Widget", price=10.99, quantity=2)
    print("  ✓", example.model_dump())
except ValidationError as error:
    print(error)

print(f"\nGenerated data directory: {DATA_DIR.resolve()}")
print(f"Logging to: {LOG_FILE.resolve()}")
logger.info("Notebook setup completed")


# Step 2: Creating Product Data Models

**Objective:** Design Pydantic models for product listing requests.

### What to do

- Define the structure of product data for your previous listing generator
- Create Pydantic models with validation rules that fit your use case
- Add custom validators for business logic

### Tips

- Consider fields like `name`, `price`, `category`, `description`, and `additional_info`
- Start simple and add complexity gradually
- Think about which fields are required and which can be optional

### Expected outcome

You should have working Pydantic models that validate product data correctly.

### Checkpoint

Verify that:

- Valid data passes validation
- Invalid data raises appropriate errors
- Required fields are enforced
- Custom validators work correctly

In [ ]:
print("=" * 60)
print("STEP 2 | DATA MODELS")
print("=" * 60)


class ProductRequest(BaseModel):
    """Validated input schema for product requests."""

    id: str = Field(min_length=2, max_length=20)
    name: str = Field(min_length=3, max_length=80)
    category: str = Field(min_length=3, max_length=40)
    price: float = Field(gt=0)
    description: str = Field(min_length=20, max_length=500)
    features: List[str] = Field(default_factory=list)
    quantity: int = Field(default=1, ge=1)
    currency: str = Field(default="USD", min_length=3, max_length=3)
    brand: Optional[str] = Field(default=None, max_length=40)
    additional_info: Optional[str] = None

    @field_validator("id", "name", "category", "description", "brand", "additional_info")
    @classmethod
    def strip_text_fields(cls, value):
        if value is None:
            return value
        value = value.strip()
        if not value:
            raise ValueError("Text fields cannot be empty")
        return value

    @field_validator("currency")
    @classmethod
    def validate_currency(cls, value: str) -> str:
        value = value.upper().strip()
        if len(value) != 3 or not value.isalpha():
            raise ValueError("Currency must be a 3-letter code such as USD or EUR")
        return value

    @field_validator("features")
    @classmethod
    def validate_features(cls, value: List[str]) -> List[str]:
        cleaned = [item.strip() for item in value if item.strip()]
        if not cleaned:
            raise ValueError("At least one feature is required")
        return cleaned


class ProductBatch(BaseModel):
    """Schema for a JSON file containing multiple products."""

    products: List[ProductRequest]


class ProductListing(BaseModel):
    """Validated output schema for generated product listings."""

    product_id: str
    title: str
    marketing_description: str
    bullet_points: List[str]
    tags: List[str]
    category: str
    price_label: str


sample_valid = {
    "id": "P001",
    "name": "EcoSmart Water Bottle",
    "category": "Kitchen",
    "price": 24.99,
    "description": "Reusable stainless steel bottle with double-wall insulation that keeps drinks cold for 24 hours.",
    "features": ["BPA-free", "Leak-proof lid", "750ml capacity"],
    "quantity": 12,
    "currency": "usd",
    "brand": "EcoSmart",
    "additional_info": "Ideal for travel, work, and gym sessions."
}

sample_invalid = {
    "id": "P002",
    "name": " ",
    "category": "Ki",
    "price": -12,
    "description": "Too short",
    "features": [],
    "quantity": 0,
    "currency": "US D"
}

print("\nModel validation test:")
try:
    validated = ProductRequest.model_validate(sample_valid)
    print("  ✓ Valid product:", validated.model_dump())
except ValidationError as error:
    print(error)

print("\nInvalid model test:")
try:
    ProductRequest.model_validate(sample_invalid)
except ValidationError as error:
    for item in error.errors():
        print(f"  - {item['loc']}: {item['msg']}")


# Step 3: Validating JSON Input

**Objective:** Validate JSON data from files.

### What to do

- Generate example JSON files using the structure from Step 2
- Create both valid and invalid examples
- Write a function to load JSON data and validate it using your Pydantic models
- Define a strategy to handle validation errors
- Return validated data when successful

### Expected outcome

You should be able to validate JSON data and get clear error messages for invalid input.

### Checkpoint

Verify that validation works for both valid and invalid JSON.

In [ ]:
print("=" * 60)
print("STEP 3 | HELPER FUNCTIONS AND FILE PREPARATION")
print("=" * 60)


def load_json_file(file_path: str) -> dict:
    """Load and parse a JSON file with explicit error handling."""

    logger.info("Loading JSON file: %s", file_path)
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError as error:
        message = build_error_message(
            "load_json_file",
            type(error).__name__,
            f"File '{file_path}' not found. Current directory: {os.getcwd()}",
            str(error),
            "Check that the file path is correct and that the file exists."
        )
        logger.error(message)
        print(message)
        raise
    except json.JSONDecodeError as error:
        message = build_error_message(
            "load_json_file",
            type(error).__name__,
            f"File '{file_path}', line {error.lineno}, column {error.colno}",
            error.msg,
            "Check the JSON syntax near the reported line and column."
        )
        logger.error(message)
        print(message)
        raise
    except OSError as error:
        message = build_error_message(
            "load_json_file",
            type(error).__name__,
            f"File '{file_path}'",
            str(error),
            "Check file permissions and try again."
        )
        logger.error(message)
        print(message)
        raise



def save_json_file(data: dict | list, file_path: str) -> None:
    """Save JSON data to disk with explicit error handling."""

    logger.info("Saving JSON file: %s", file_path)
    try:
        with open(file_path, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2)
    except OSError as error:
        message = build_error_message(
            "save_json_file",
            type(error).__name__,
            f"File '{file_path}'",
            str(error),
            "Check the output path and write permissions."
        )
        logger.error(message)
        print(message)
        raise



def validate_product_data(product_dict: dict) -> Optional[ProductRequest]:
    """Validate a product dictionary and show which fields failed."""

    try:
        product = ProductRequest.model_validate(product_dict)
        logger.info("Validated product: %s", product.id)
        return product
    except ValidationError as error:
        lines = []
        for item in error.errors():
            lines.append(f"    - {item['loc']}: {item['msg']}")
        message = build_error_message(
            "validate_product_data",
            type(error).__name__,
            f"Product ID: {product_dict.get('id', 'unknown')}",
            "\n".join(lines),
            "Fix the invalid fields shown above and run validation again."
        )
        logger.error(message)
        print(message)
        return None



def create_product_prompt(product: ProductRequest) -> str:
    """Generate a prompt for one product."""

    features_text = ", ".join(product.features)
    brand_text = product.brand or "Unknown brand"
    additional_text = product.additional_info or "No additional information provided."
    return (
        f"Create a polished e-commerce product listing for the following item. "
        f"Product ID: {product.id}. "
        f"Name: {product.name}. "
        f"Category: {product.category}. "
        f"Price: {product.price:.2f} {product.currency}. "
        f"Brand: {brand_text}. "
        f"Description: {product.description}. "
        f"Features: {features_text}. "
        f"Additional info: {additional_text}."
    )



def parse_api_response(response) -> str:
    """Extract plain text from a chat completion response."""

    try:
        return response.choices[0].message.content
    except (AttributeError, IndexError, KeyError, TypeError) as error:
        message = build_error_message(
            "parse_api_response",
            type(error).__name__,
            "OpenAI chat completion response object",
            str(error),
            "Inspect the response structure and confirm choices[0].message.content exists."
        )
        logger.error(message)
        print(message)
        raise



def format_output(product: ProductRequest, listing: ProductListing) -> dict:
    """Format the final saved result for one product."""

    return {
        "product_id": product.id,
        "name": product.name,
        "input": product.model_dump(),
        "output": listing.model_dump()
    }



def load_and_validate_products(json_path: str) -> List[ProductRequest]:
    """Load a product file and validate each product."""

    payload = load_json_file(json_path)
    raw_products = payload.get("products", [])
    if not isinstance(raw_products, list):
        message = build_error_message(
            "load_and_validate_products",
            "TypeError",
            f"File '{json_path}'",
            "Expected a 'products' list in the JSON payload.",
            "Use the structure {'products': [...]} for batch files."
        )
        logger.error(message)
        print(message)
        raise TypeError(message)

    validated_products = []
    for item in raw_products:
        product = validate_product_data(item)
        if product is not None:
            validated_products.append(product)

    logger.info("Validated %s/%s products from %s", len(validated_products), len(raw_products), json_path)
    return validated_products


products_payload = {
    "products": [
        {
            "id": "P001",
            "name": "Wireless Bluetooth Headphones",
            "category": "Electronics",
            "price": 99.99,
            "description": "Wireless headphones with active noise cancellation, deep bass, and long battery life for travel and daily work.",
            "features": ["Bluetooth 5.0", "Noise Cancelling", "30hr Battery", "Comfortable Fit"],
            "quantity": 8,
            "currency": "USD",
            "brand": "SoundMax",
            "additional_info": "Includes carrying case and USB-C charging cable."
        },
        {
            "id": "P002",
            "name": "Smart Watch",
            "category": "Wearables",
            "price": 249.99,
            "description": "Smart watch with fitness tracking, GPS, heart rate monitoring, and a bright all-day display.",
            "features": ["Heart Rate Monitor", "GPS", "Water Resistant", "Sleep Tracking"],
            "quantity": 5,
            "currency": "USD",
            "brand": "PulseTrack",
            "additional_info": "Compatible with Android and iOS."
        },
        {
            "id": "P003",
            "name": "Laptop Stand",
            "category": "Accessories",
            "price": 49.99,
            "description": "Adjustable aluminum stand that improves posture, cooling, and desk organization for laptop users.",
            "features": ["Adjustable Height", "Aluminum Construction", "Cable Management"],
            "quantity": 14,
            "currency": "EUR",
            "brand": "DeskFlow",
            "additional_info": "Folds flat for easy storage."
        }
    ]
}

invalid_products_payload = {
    "products": [
        {
            "id": "P004",
            "name": "X",
            "category": "",
            "price": -10,
            "description": "short text",
            "features": [],
            "quantity": 0,
            "currency": "US D"
        }
    ]
}

save_json_file(products_payload, DATA_DIR / "products.json")
save_json_file(invalid_products_payload, DATA_DIR / "invalid_products.json")
with open(DATA_DIR / "malformed.json", "w", encoding="utf-8") as file:
    file.write('{"products": [ {"id": "BROKEN", "name": "Oops" }')

print("Created test files in data/: data/products.json, data/invalid_products.json, data/malformed.json")


# Step 4: Integrating with ChatGPT API

**Objective:** Combine validation with your existing ChatGPT API workflow.

### What to do

- Import or reuse your previous ChatGPT/OpenAI API code
- Validate input before processing
- Process only validated requests
- Return appropriate responses
- Validate that the output is also in a standardized format, for example with Pydantic models

### Expected outcome

You should have a complete workflow that validates the input before processing and validates the output after processing.

### Checkpoint

Verify that:

- Invalid data is rejected before API calls
- Valid data is processed successfully
- Errors are handled gracefully

In [ ]:
print("=" * 60)
print("STEP 4 | OPENAI WRAPPER WITH RETRY LOGIC")
print("=" * 60)


class OpenAIWrapper:
    """Reusable OpenAI wrapper with retry logic and structured output support."""

    def __init__(self, api_key: Optional[str], max_retries: int = 3, timeout: int = 30):
        self.api_key = api_key
        self.max_retries = max_retries
        self.timeout = timeout
        self.client = OpenAI(api_key=api_key, timeout=timeout) if api_key else None

    def _fallback_listing(self, product: ProductRequest) -> ProductListing:
        """Return a local demo listing when an API key is unavailable."""

        logger.info("Using fallback listing for product %s because OPENAI_API_KEY is not set", product.id)
        return ProductListing(
            product_id=product.id,
            title=f"{product.brand + ' ' if product.brand else ''}{product.name}",
            marketing_description=(
                f"{product.name} is a {product.category.lower()} product built for customers who want "
                f"practical features, reliable quality, and clear everyday value."
            ),
            bullet_points=product.features[:4],
            tags=[product.category.lower(), product.currency.lower(), "refactored-pipeline"],
            category=product.category,
            price_label=f"{product.currency} {product.price:.2f}"
        )

    def generate_listing(self, product: ProductRequest, model: str = DEFAULT_MODEL) -> ProductListing:
        """Generate a structured listing with retries and explicit API error reporting."""

        if self.client is None:
            return self._fallback_listing(product)

        prompt = create_product_prompt(product)
        for attempt in range(1, self.max_retries + 1):
            try:
                logger.info("Calling OpenAI for product %s (attempt %s/%s)", product.id, attempt, self.max_retries)
                completion = self.client.chat.completions.parse(
                    model=model,
                    messages=[
                        {
                            "role": "system",
                            "content": "You create concise, high-quality e-commerce listings and return structured data only."
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    response_format=ProductListing,
                )
                listing = completion.choices[0].message.parsed
                logger.info("Received structured listing for product %s", product.id)
                return listing
            except (APIConnectionError, APITimeoutError, APIError) as error:
                if attempt < self.max_retries:
                    wait_time = 2 ** (attempt - 1)
                    logger.warning(
                        "API call failed for %s on attempt %s/%s. Retrying in %s seconds.",
                        product.id,
                        attempt,
                        self.max_retries,
                        wait_time,
                    )
                    time.sleep(wait_time)
                    continue

                location = f"Product {product.name} (ID: {product.id})"
                message = build_error_message(
                    "OpenAIWrapper.generate_listing",
                    type(error).__name__,
                    location,
                    str(error),
                    "Check the API key, network connection, timeout settings, and rate limits."
                )
                logger.error(message)
                print(message)
                raise RuntimeError(message) from error
            except Exception as error:
                trace = traceback.format_exc(limit=1)
                message = build_error_message(
                    "OpenAIWrapper.generate_listing",
                    type(error).__name__,
                    f"Product {product.name} (ID: {product.id})",
                    f"{error} | Trace: {trace.strip()}",
                    "Inspect the response format and model compatibility."
                )
                logger.error(message)
                print(message)
                raise RuntimeError(message) from error


api_wrapper = OpenAIWrapper(api_key=os.getenv("OPENAI_API_KEY"), max_retries=3, timeout=30)
valid_products = load_and_validate_products(DATA_DIR / "products.json")
demo_listing = api_wrapper.generate_listing(valid_products[0])
print("\nExample structured listing:")
print(demo_listing.model_dump_json(indent=2))


# Step 5: Handling Multiple Requests

**Objective:** Process a list of incoming JSON requests in one run.

### What to do

- Load a list of request payloads from a JSON file
- Validate each payload with your Pydantic model
- Process valid requests
- Collect structured errors for invalid ones
- Return a summary of successes and failures

### Expected outcome

You can process batches while preserving clear per-request validation feedback.

### Checkpoint

Verify that your batch run handles mixed valid and invalid requests without stopping the full pipeline.

In [ ]:
print("=" * 60)
print("STEP 5 | MODULAR PROCESSING PIPELINE")
print("=" * 60)


def generate_description(product: ProductRequest, api_client: OpenAIWrapper) -> ProductListing:
    """Generate one product listing with context-rich error handling."""

    try:
        return api_client.generate_listing(product)
    except Exception as error:
        message = build_error_message(
            "generate_description",
            type(error).__name__,
            f"Product {product.name} (ID: {product.id})",
            str(error),
            "Review the API wrapper logs and verify the product input data."
        )
        logger.error(message)
        print(message)
        raise RuntimeError(message) from error



def process_products(products: List[ProductRequest], api_client: OpenAIWrapper) -> List[dict]:
    """Process all products and keep explicit success/error records."""

    results = []
    for product in products:
        try:
            listing = generate_description(product, api_client)
            results.append({
                "status": "success",
                "product_id": product.id,
                "result": format_output(product, listing)
            })
        except Exception as error:
            results.append({
                "status": "error",
                "product_id": product.id,
                "error": str(error)
            })
    return results



def save_results(results: List[dict], output_path: str) -> None:
    """Save pipeline results to disk."""

    save_json_file(results, output_path)
    logger.info("Saved %s results to %s", len(results), output_path)



def run_product_pipeline(input_path: str, output_path: str, api_client: OpenAIWrapper) -> dict:
    """Full orchestration function for the product pipeline."""

    logger.info("Starting pipeline for input=%s output=%s", input_path, output_path)
    products = load_and_validate_products(input_path)
    results = process_products(products, api_client)
    save_results(results, output_path)

    summary = {
        "input_file": input_path,
        "output_file": output_path,
        "total_products": len(products),
        "success_count": sum(1 for item in results if item["status"] == "success"),
        "error_count": sum(1 for item in results if item["status"] == "error")
    }
    logger.info("Pipeline summary: %s", summary)
    return {"summary": summary, "results": results}


pipeline_run = run_product_pipeline(DATA_DIR / "products.json", DATA_DIR / "results_refactored.json", api_wrapper)
print(json.dumps(pipeline_run["summary"], indent=2))
print("\nFirst pipeline result:")
print(json.dumps(pipeline_run["results"][0], indent=2))


# Step 6: Creating a Client Request Handler

**Objective:** Wrap validation and processing into a reusable handler function.

### What to do

- Create a `handle_client_request(payload: dict)` function
- Validate input
- Call your processing logic
- Validate output
- Return a consistent response shape for success and error cases

### Expected outcome

You have one reusable entry point for future API endpoints or scripts.

### Checkpoint

Verify that the handler returns predictable JSON for both valid and invalid payloads.

In [ ]:
print("=" * 60)
print("STEP 6 | CLIENT REQUEST HANDLER AND TESTS")
print("=" * 60)


def handle_client_request(payload: dict, api_client: OpenAIWrapper) -> dict:
    """Reusable entry point for validating one request and generating one response."""

    try:
        product = validate_product_data(payload)
        if product is None:
            return {
                "status": "error",
                "message": "Input validation failed",
                "data": None,
                "errors": ["See validation messages above for field-level details."]
            }

        listing = generate_description(product, api_client)
        return {
            "status": "success",
            "message": "Product listing generated successfully",
            "data": format_output(product, listing),
            "errors": []
        }
    except Exception as error:
        trace = traceback.format_exc(limit=1)
        message = build_error_message(
            "handle_client_request",
            type(error).__name__,
            f"Payload for product ID {payload.get('id', 'unknown')}",
            f"{error} | Trace: {trace.strip()}",
            "Inspect the payload structure and the earlier pipeline logs."
        )
        logger.error(message)
        print(message)
        return {
            "status": "error",
            "message": "Request processing failed",
            "data": None,
            "errors": [message]
        }


single_valid_payload = products_payload["products"][0]
single_invalid_payload = invalid_products_payload["products"][0]

print("\nValid request result:")
print(json.dumps(handle_client_request(single_valid_payload, api_wrapper), indent=2))

print("\nInvalid request result:")
print(json.dumps(handle_client_request(single_invalid_payload, api_wrapper), indent=2))

print("\nMissing file test:")
try:
    load_json_file(DATA_DIR / "missing.json")
except FileNotFoundError:
    print("Expected missing file error captured.\n")

print("Malformed JSON test:")
try:
    load_json_file(DATA_DIR / "malformed.json")
except json.JSONDecodeError:
    print("Expected malformed JSON error captured.\n")

print("Invalid product batch test:")
invalid_batch_products = load_and_validate_products(DATA_DIR / "invalid_products.json")
print(f"Validated products found in invalid batch: {len(invalid_batch_products)}")


# Bonus Challenge: New Dataset from Scratch

**Objective:** Apply your validation skills to a completely new dataset.

### Task

Choose a new dataset from Kaggle, Hugging Face, or create your own. Then:

- Analyze the dataset structure
- Identify field names, types, and sensible validation rules
- Create Pydantic models for the dataset
- Build a validation pipeline
- Validate each record and report results
- Use validated data for some purpose such as analysis or API calls
- Integrate validated data with ChatGPT/OpenAI
- Validate the model output as well

### Suggested datasets

- Customer Reviews Dataset
- Real Estate Listings
- Employee Records
- E-commerce Orders

In [ ]:
print("=" * 60)
print("ADVANCED | LOGGING AND REFACTORING SUMMARY")
print("=" * 60)


summary_notes = {
    "refactoring_improvements": [
        "Split the monolithic flow into helper functions with single responsibilities.",
        "Added explicit file, JSON, validation, and API error handling.",
        "Introduced OpenAIWrapper with retry logic and fallback behavior.",
        "Separated orchestration from business logic in run_product_pipeline().",
        "Added structured logging to both console and file."
    ],
    "generated_files": [
        str(DATA_DIR / "products.json"),
        str(DATA_DIR / "invalid_products.json"),
        str(DATA_DIR / "malformed.json"),
        str(DATA_DIR / "results_refactored.json"),
        str(LOG_FILE)
    ]
}
print(json.dumps(summary_notes, indent=2))

log_path = Path(LOG_FILE)
if log_path.exists():
    print("\nLast log lines:")
    log_lines = log_path.read_text(encoding="utf-8").splitlines()
    for line in log_lines[-10:]:
        print(line)
else:
    print("Log file not found.")


# Deliverables

## Required

- Your complete `api_json_validation.py` file with:
  - Pydantic models
  - Validation functions
  - Integration with ChatGPT API
  - Client request handler
- Example JSON files with valid and invalid examples
- Screenshots or output showing:
  - Successful validation
  - Validation errors
  - Complete workflow execution
- A brief report including:
  - How Pydantic validation works
  - Benefits of validation
  - Challenges you faced
  - Improvements you made

## Bonus

- Pydantic models for a new dataset
- Validation results and statistics
- A brief analysis report

# Troubleshooting Notes

## Common issues

### 1. Validation errors are unclear
- Use `try/except` to catch `ValidationError`
- Use `error.errors()` for detailed error information

### 2. Optional fields are required
- Use `Optional[type] = None`
- Or use `Field(default=None)`

### 3. Custom validator is not working
- Check the `@validator('field_name')` decorator
- Check the validator function signature
- Use `pre=True` if needed before type conversion

### 4. JSON parsing fails before validation
- Validate raw JSON format first with `json.loads()`
- Handle `JSONDecodeError` separately

### 5. Validation is too strict or too lenient
- Adjust `Field` constraints such as `min_length`, `max_length`, `gt`, and `lt`
- Modify custom validators as needed

### 6. Integration with existing code is difficult
- Convert Pydantic models to dictionaries with `.dict()`
- Use `.json()` for serialization
- Access fields as attributes such as `product.name`

# To-Do Checklist

- Complete Step 1: Setting Up Pydantic
- Complete Step 2: Creating Product Data Models
- Complete Step 3: Validating JSON Input
- Complete Step 4: Integrating with ChatGPT API
- Complete Step 5: Handling Multiple Requests
- Complete Step 6: Creating a Client Request Handler
- Test with various valid and invalid inputs
- Document your validation rules
- Optionally complete the bonus challenge
- Submit your work

# Reflection

After completing the lab, reflect on the following:

- Why is validation important before processing data?
- What advantages does Pydantic provide over manual validation?
- How does validation improve error messages and debugging?
- How do type hints and Pydantic work together?
- How does validation improve API reliability?
- Which validation patterns were most useful?

## Key takeaways

- Validation prevents errors and improves data quality
- Pydantic provides automatic validation with clear error messages
- Type hints make code more maintainable and less error-prone
- Validating early saves time and resources
- Good validation improves user experience
- Validation is essential for production systems